### Data and function preperation

In [ ]:
### pip install numpy/ panda/ reverse_geocode/ geopy/ folium/ requests/ geopandas/ shapelyd5

In [54]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datetime import datetime
import reverse_geocode as rg
from geopy.distance import geodesic
import folium
from folium import GeoJson
import os
from IPython.display import display, HTML, IFrame
import requests
import geopandas as gpd
from shapely.geometry import shape, Point, LineString
import sys

In [2]:
df = pd.read_csv("Checkins_data.txt", sep ='\t', header = None,
                 names=["user_id", "checkin_time", "lat_org", "lon_org", "location_id"])
df['checkin_time'] = pd.to_datetime(df['checkin_time'])
distinct_location_ids = df.drop_duplicates(subset=["lat_org", "lon_org"])
# distinct_location_ids["coordinates"] = distinct_location_ids["lat"].apply(str)+ " , " +distinct_location_ids["lon"].apply(str)
distinct_location_ids['coordinates'] = distinct_location_ids.apply(lambda row: (row['lat_org'], row['lon_org']), axis=1)

distinct_user_location = (
    df.groupby('user_id')['location_id']
    .apply(lambda x: sorted(x.unique()))  # Extract unique location_ids for each user_id
    .reset_index()
)

# Rename columns for clarity
distinct_user_location.rename(columns={'location_id': 'distinct_user_location'}, inplace=True)

results = rg.search(list(distinct_location_ids['coordinates']))

# Convert results into a DataFrame
location_details = pd.json_normalize(results)

# Merge back into the original DataFrame
distinct_location_ids = pd.concat([distinct_location_ids.reset_index(drop=True), location_details], axis=1)

# Create user last check-in table
last_checkin = df.loc[df.groupby('user_id')['checkin_time'].idxmax()]
last_checkin = last_checkin[['user_id', 'lat_org',	'lon_org', 'location_id']]
last_checkin.head(5)

# Generate initial clusters
def generate_cluster_key(row):
    if pd.isna(row['state']):  # If 'state' is missing, cluster by 'country' only
        return row['country']
    else:  # Cluster by 'country + state'
        return f"{row['country']} - {row['state']}"

distinct_location_ids['cluster'] = distinct_location_ids.apply(generate_cluster_key, axis=1)

# Count data points in each cluster
cluster_counts = distinct_location_ids['cluster'].value_counts().to_dict()

# Reassign clusters for those with fewer than 15 data points
def reassign_low_count_cluster(row):
    cluster = row['cluster']
    if cluster_counts[cluster] > 15:  # Keep original cluster if count > 15
        return cluster
    elif pd.isna(row['state']):  # Country-level cluster (no state available)
        return row['country']
    else:  # Re-cluster by 'country + state'
        state_cluster = f"{row['country']} - {row['state']}"
        state_cluster_count = cluster_counts.get(state_cluster, 0)
        if state_cluster_count > 15:
            return state_cluster
        # Re-cluster by 'country' if 'country + state' still doesn't meet threshold
        return row['country']

COUNTRY_ALPHA2_TO_CONTINENT = {
    'AB': 'Asia',
    'AD': 'Europe',
    'AE': 'Asia',
    'AF': 'Asia',
    'AG': 'North America',
    'AI': 'North America',
    'AL': 'Europe',
    'AM': 'Asia',
    'AO': 'Africa',
    'AR': 'South America',
    'AS': 'Oceania',
    'AT': 'Europe',
    'AU': 'Oceania',
    'AW': 'North America',
    'AX': 'Europe',
    'AZ': 'Asia',
    'BA': 'Europe',
    'BB': 'North America',
    'BD': 'Asia',
    'BE': 'Europe',
    'BF': 'Africa',
    'BG': 'Europe',
    'BH': 'Asia',
    'BI': 'Africa',
    'BJ': 'Africa',
    'BL': 'North America',
    'BM': 'North America',
    'BN': 'Asia',
    'BO': 'South America',
    'BQ': 'North America',
    'BR': 'South America',
    'BS': 'North America',
    'BT': 'Asia',
    'BV': 'Antarctica',
    'BW': 'Africa',
    'BY': 'Europe',
    'BZ': 'North America',
    'CA': 'North America',
    'CC': 'Asia',
    'CD': 'Africa',
    'CF': 'Africa',
    'CG': 'Africa',
    'CH': 'Europe',
    'CI': 'Africa',
    'CK': 'Oceania',
    'CL': 'South America',
    'CM': 'Africa',
    'CN': 'Asia',
    'CO': 'South America',
    'CR': 'North America',
    'CU': 'North America',
    'CV': 'Africa',
    'CW': 'North America',
    'CX': 'Asia',
    'CY': 'Asia',
    'CZ': 'Europe',
    'DE': 'Europe',
    'DJ': 'Africa',
    'DK': 'Europe',
    'DM': 'North America',
    'DO': 'North America',
    'DZ': 'Africa',
    'EC': 'South America',
    'EE': 'Europe',
    'EG': 'Africa',
    'ER': 'Africa',
    'ES': 'Europe',
    'ET': 'Africa',
    'FI': 'Europe',
    'FJ': 'Oceania',
    'FK': 'South America',
    'FM': 'Oceania',
    'FO': 'Europe',
    'FR': 'Europe',
    'GA': 'Africa',
    'GB': 'Europe',
    'GD': 'North America',
    'GE': 'Asia',
    'GF': 'South America',
    'GG': 'Europe',
    'GH': 'Africa',
    'GI': 'Europe',
    'GL': 'North America',
    'GM': 'Africa',
    'GN': 'Africa',
    'GP': 'North America',
    'GQ': 'Africa',
    'GR': 'Europe',
    'GS': 'South America',
    'GT': 'North America',
    'GU': 'Oceania',
    'GW': 'Africa',
    'GY': 'South America',
    'HK': 'Asia',
    'HM': 'Antarctica',
    'HN': 'North America',
    'HR': 'Europe',
    'HT': 'North America',
    'HU': 'Europe',
    'ID': 'Asia',
    'IE': 'Europe',
    'IL': 'Asia',
    'IM': 'Europe',
    'IN': 'Asia',
    'IO': 'Asia',
    'IQ': 'Asia',
    'IR': 'Asia',
    'IS': 'Europe',
    'IT': 'Europe',
    'JE': 'Europe',
    'JM': 'North America',
    'JO': 'Asia',
    'JP': 'Asia',
    'KE': 'Africa',
    'KG': 'Asia',
    'KH': 'Asia',
    'KI': 'Oceania',
    'KM': 'Africa',
    'KN': 'North America',
    'KP': 'Asia',
    'KR': 'Asia',
    'KW': 'Asia',
    'KY': 'North America',
    'KZ': 'Asia',
    'LA': 'Asia',
    'LB': 'Asia',
    'LC': 'North America',
    'LI': 'Europe',
    'LK': 'Asia',
    'LR': 'Africa',
    'LS': 'Africa',
    'LT': 'Europe',
    'LU': 'Europe',
    'LV': 'Europe',
    'LY': 'Africa',
    'MA': 'Africa',
    'MC': 'Europe',
    'MD': 'Europe',
    'ME': 'Europe',
    'MF': 'North America',
    'MG': 'Africa',
    'MH': 'Oceania',
    'MK': 'Europe',
    'ML': 'Africa',
    'MM': 'Asia',
    'MN': 'Asia',
    'MO': 'Asia',
    'MP': 'Oceania',
    'MQ': 'North America',
    'MR': 'Africa',
    'MS': 'North America',
    'MT': 'Europe',
    'MU': 'Africa',
    'MV': 'Asia',
    'MW': 'Africa',
    'MX': 'North America',
    'MY': 'Asia',
    'MZ': 'Africa',
    'NA': 'Africa',
    'NC': 'Oceania',
    'NE': 'Africa',
    'NF': 'Oceania',
    'NG': 'Africa',
    'NI': 'North America',
    'NL': 'Europe',
    'NO': 'Europe',
    'NP': 'Asia',
    'NR': 'Oceania',
    'NU': 'Oceania',
    'NZ': 'Oceania',
    'OM': 'Asia',
    'OS': 'Asia',
    'PA': 'North America',
    'PE': 'South America',
    'PF': 'Oceania',
    'PG': 'Oceania',
    'PH': 'Asia',
    'PK': 'Asia',
    'PL': 'Europe',
    'PM': 'North America',
    'PR': 'North America',
    'PS': 'Asia',
    'PT': 'Europe',
    'PW': 'Oceania',
    'PY': 'South America',
    'QA': 'Asia',
    'RE': 'Africa',
    'RO': 'Europe',
    'RS': 'Europe',
    'RU': 'Europe',
    'RW': 'Africa',
    'SA': 'Asia',
    'SB': 'Oceania',
    'SC': 'Africa',
    'SD': 'Africa',
    'SE': 'Europe',
    'SG': 'Asia',
    'SH': 'Africa',
    'SI': 'Europe',
    'SJ': 'Europe',
    'SK': 'Europe',
    'SL': 'Africa',
    'SM': 'Europe',
    'SN': 'Africa',
    'SO': 'Africa',
    'SR': 'South America',
    'SS': 'Africa',
    'ST': 'Africa',
    'SV': 'North America',
    'SX': 'North America',
    'SY': 'Asia',
    'SZ': 'Africa',
    'TC': 'North America',
    'TD': 'Africa',
    'TG': 'Africa',
    'TH': 'Asia',
    'TJ': 'Asia',
    'TK': 'Oceania',
    'TM': 'Asia',
    'TN': 'Africa',
    'TO': 'Oceania',
    'TP': 'Asia',
    'TR': 'Asia',
    'TT': 'North America',
    'TV': 'Oceania',
    'TW': 'Asia',
    'TZ': 'Africa',
    'UA': 'Europe',
    'UG': 'Africa',
    'US': 'North America',
    'UY': 'South America',
    'UZ': 'Asia',
    'VC': 'North America',
    'VE': 'South America',
    'VG': 'North America',
    'VA': 'Europe',
    'VI': 'North America',
    'VN': 'Asia',
    'VU': 'Oceania',
    'WF': 'Oceania',
    'WS': 'Oceania',
    'XK': 'Europe',
    'YE': 'Asia',
    'YT': 'Africa',
    'ZA': 'Africa',
    'ZM': 'Africa',
    'ZW': 'Africa',
}


def convert_country_alpha2_to_continent(country_2_code):
    """Convert country code to continent.
    """
    if country_2_code not in COUNTRY_ALPHA2_TO_CONTINENT:
        raise KeyError

    return COUNTRY_ALPHA2_TO_CONTINENT[country_2_code]

def get_continent_from_country_code(country_code):
    """
    Get the continent name based on the ISO Alpha-2 country code.

    Parameters:
        country_code (str): ISO Alpha-2 country code (e.g., 'US', 'FR').

    Returns:
        str: Continent name (e.g., 'North America', 'Europe') or 'Unknown' if not found.
    """
    try:
        # Convert ISO Alpha-2 country code to continent code
        continent_code = convert_country_alpha2_to_continent(country_code)

        return continent_code
    except Exception:
        return "Unknown"

# Apply function to the DataFrame
distinct_location_ids['continent'] = distinct_location_ids['country_code'].apply(get_continent_from_country_code)

# Apply reassignment
distinct_location_ids['revised_cluster'] = distinct_location_ids.apply(reassign_low_count_cluster, axis=1)

# Find user last check-in location cluster
last_checkin = last_checkin.merge(distinct_location_ids[['lat_org',	'lon_org', 'location_id','country','continent','revised_cluster']],
                                  on=['lat_org',	'lon_org', 'location_id'],
                                  how='left')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18184\2945113637.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distinct_location_ids['coordinates'] = distinct_location_ids.apply(lambda row: (row['lat_org'], row['lon_org']), axis=1)


In [3]:
def recommend_unvisited_locations(
    user_id, 
    last_checkin, 
    distinct_user_location, 
    distinct_location_ids, 
    num_recommendations=15
):
    """
    Recommend unvisited locations for a given user, filtering based on cluster hierarchy.
    
    Parameters:
        user_id (int): The ID of the user.
        last_checkin (DataFrame): Last check-in table containing user's current location and cluster info.
        distinct_user_location (DataFrame): Table with all distinct locations visited by users.
        distinct_location_ids (DataFrame): Table with all distinct locations and their clusters.
        num_recommendations (int): Number of recommendations to return.
    
    Returns:
        DataFrame: Top nearest unvisited locations.
    """
    # Validate inputs
    if last_checkin.empty or distinct_location_ids.empty:
        return pd.DataFrame(columns=["location_id", "lat_org", "lon_org", "distance"])

    # Get the user's last check-in details
    user_checkin = last_checkin.loc[last_checkin["user_id"] == user_id]
    if user_checkin.empty:
        return pd.DataFrame(columns=["location_id", "lat_org", "lon_org", "distance"])

    # Extract user location details
    user_lat = user_checkin["lat_org"].values[0]
    user_lon = user_checkin["lon_org"].values[0]
    revised_cluster = user_checkin["revised_cluster"].values[0]
    country = user_checkin["country"].values[0]
    continent = user_checkin["continent"].values[0]

    # Get user's visited locations
    visited_locations = distinct_user_location.loc[
        distinct_user_location["user_id"] == user_id, "distinct_user_location"
    ]
    visited_locations = visited_locations.values[0] if not visited_locations.empty else []

    # Filtering hierarchy
    filtering_levels = [
        {"type": "cluster", "condition": distinct_location_ids["revised_cluster"] == revised_cluster},
        {"type": "country", "condition": distinct_location_ids["country"] == country},
        {"type": "continent", "condition": distinct_location_ids["continent"] == continent}
    ]

    # Find unvisited locations
    for level in filtering_levels:
        filtered_locations = distinct_location_ids[level["condition"]]
        unvisited_locations = filtered_locations[~filtered_locations["location_id"].isin(visited_locations)].copy()
        
        if len(unvisited_locations) >= num_recommendations:
            break

    # If still no locations found, return empty DataFrame
    if len(unvisited_locations) == 0:
        return pd.DataFrame(columns=["location_id", "lat_org", "lon_org", "distance"])

    # Optimized distance calculation
    user_coords = (user_lat, user_lon)
    unvisited_locations["distance"] = unvisited_locations.apply(
        lambda row: geodesic(user_coords, (row["lat_org"], row["lon_org"])).kilometers, 
        axis=1
    )

    # Sort by distance and select top recommendations
    recommendations = unvisited_locations.nsmallest(num_recommendations, "distance")
    return recommendations[["location_id", "lat_org", "lon_org", "distance"]]

In [4]:
social_network = pd.read_csv("Social_Network.txt", sep ='\t', header = None,
                 names=["user_id", "friend_id"])

social_network = social_network.groupby('user_id')['friend_id'].apply(lambda x: ', '.join(map(str, x))).reset_index()

# Rename the column for clarity
social_network.rename(columns={'friend': 'friends'}, inplace=True)

# Output
print(social_network)

      user_id                                          friend_id
0           2  22, 53, 111, 185, 205, 209, 211, 218, 222, 241...
1          22  2, 111, 241, 247, 267, 341, 381, 395, 531, 540...
2          35                        491, 2810, 4632, 4994, 5385
3          53  2, 111, 124, 130, 241, 322, 382, 395, 401, 444...
4          54                     143, 267, 854, 956, 3682, 6220
...       ...                                                ...
1025   129893                                             129894
1026   129894                                             129893
1027   138741                                     103924, 110523
1028   143249                                              10616
1029   179156                                             117729

[1030 rows x 2 columns]


In [5]:
# Convert 'friends' column (comma-separated) to lists
social_network['friend_id'] = social_network['friend_id'].apply(lambda x: list(map(int, x.split(','))))

# Define a function to get 2-hop friends
def get_2hop_friends(user_id, social_network):
    # Find the direct friends of the user
    direct_friends = social_network.loc[social_network['user_id'] == user_id, 'friend_id'].values
    if len(direct_friends) == 0:
        return []
    direct_friends = direct_friends[0]

    # Find friends of direct friends (2-hop friends)
    two_hop_friends = set()
    for friend in direct_friends:
        friends_of_friend = social_network.loc[social_network['user_id'] == friend, 'friend_id'].values
        if len(friends_of_friend) > 0:
            two_hop_friends.update(friends_of_friend[0])

    # Exclude the original user and direct friends
    two_hop_friends.discard(user_id)
    two_hop_friends -= set(direct_friends)

    return list(two_hop_friends)

# Create a table of 2-hop friends and their locations
result = []

for user_id in social_network['user_id']:
    # Get 2-hop friends
    two_hop_friends = get_2hop_friends(user_id, social_network)

    # Get locations of 2-hop friends
    locations = []
    for friend_id in two_hop_friends:
        friend_location = distinct_user_location.loc[
            distinct_user_location['user_id'] == friend_id, 'distinct_user_location'
        ].values
        if len(friend_location) > 0:
            locations.extend(friend_location[0])

    # Append result
    result.append({'user_id': user_id, '2hop_friends_locations': locations})

# Create DataFrame
two_hop_locations = pd.DataFrame(result)

In [6]:
def recommend_unvisited_locations_2hop_friends(user_id, last_checkin, distinct_user_location, distinct_location_ids, two_hop_locations, num_recommendations=15):
    """
    Recommend unvisited locations for a given user, filtering based on cluster hierarchy.

    Parameters:
        user_id (int): The ID of the user.
        last_checkin (DataFrame): Last check-in table containing user's current location and cluster info.
        distinct_user_location (DataFrame): Table with all distinct locations visited by users.
        two_hop_locations (DataFrame): Table with user_id and their 2-hop friends' locations.
        distinct_location_ids (DataFrame): Table with all distinct locations and their clusters.
        num_recommendations (int): Number of recommendations to return.

    Returns:
        DataFrame: Top nearest unvisited locations.
    """
    # Get the user's last check-in details
    user_checkin = last_checkin.loc[last_checkin["user_id"] == user_id]
    if user_checkin.empty:
        return pd.DataFrame()  # Return empty if user not found
    user_lat = user_checkin["lat_org"].values[0]
    user_lon = user_checkin["lon_org"].values[0]
    revised_cluster = user_checkin["revised_cluster"].values[0]

    # Get user's visited locations
    visited_locations = distinct_user_location.loc[
        distinct_user_location["user_id"] == user_id, "distinct_user_location"
    ]
    visited_locations = visited_locations.values[0] if not visited_locations.empty else []

    # Get locations visited by 2-hop friends
    two_hop_friends_locations = two_hop_locations.loc[
        two_hop_locations["user_id"] == user_id, "2hop_friends_locations"
    ]
    two_hop_friends_locations = two_hop_friends_locations.values[0] if not two_hop_friends_locations.empty else []

    # Filter by 2-hop friends locations 
    two_hop_friends_locations_filter = distinct_location_ids[
        distinct_location_ids["location_id"].isin(two_hop_friends_locations)
    ]
    
    # Filter by revised_cluster
    filtered_locations = two_hop_friends_locations_filter[
        two_hop_friends_locations_filter["revised_cluster"] == revised_cluster
    ]

    # Find unvisited locations
    unvisited_locations = filtered_locations[~filtered_locations["location_id"].isin(visited_locations)].copy()

    if unvisited_locations.empty:
        return "No locations fulfill the conditions"

    # If not enough data points, broaden to country
    if len(unvisited_locations) < num_recommendations:
        filtered_locations = two_hop_friends_locations_filter[two_hop_friends_locations_filter["country"] == revised_cluster]
        unvisited_locations = filtered_locations[~filtered_locations["location_id"].isin(visited_locations)].copy()

    # If still not enough data points, broaden to continent
    if len(unvisited_locations) < num_recommendations:
        filtered_locations = two_hop_friends_locations_filter[two_hop_friends_locations_filter["continent"] == revised_cluster]
        unvisited_locations = filtered_locations[~filtered_locations["location_id"].isin(visited_locations)].copy()

    # Calculate distances for unvisited locations
    user_coords = (user_lat, user_lon)
    unvisited_locations["distance"] = unvisited_locations.apply(
        lambda row: geodesic(user_coords, (row["lat_org"], row["lon_org"])).kilometers, axis=1
    )

    # Sort by distance and select the top recommendations
    two_hop_recommendations = unvisited_locations.nsmallest(num_recommendations, "distance")

    return two_hop_recommendations[["location_id", "lat_org", "lon_org", "distance"]]

In [7]:
def get_fastest_locations(user_id, last_checkin, recommendations, mapbox_token):
    """
    Find the 10 fastest locations to reach using Mapbox Matrix API.

    Parameters:
        user_id (int): The ID of the user.
        last_checkin (DataFrame): DataFrame with user's last check-in data.
        recommendations (DataFrame): DataFrame of recommended locations.
        mapbox_token (str): Mapbox API access token.

    Returns:
        DataFrame: DataFrame of the 10 fastest locations with travel times.
    """
    # Get user's last check-in location
    user_checkin = last_checkin.loc[last_checkin["user_id"] == user_id]
    if user_checkin.empty:
        raise ValueError(f"No last check-in data found for user_id: {user_id}")
    
    user_lat = user_checkin["lat_org"].values[0]
    user_lon = user_checkin["lon_org"].values[0]

    # Prepare coordinates for the Mapbox Matrix API
    destinations = recommendations[["lat_org", "lon_org"]].values
    coordinates = f"{user_lon},{user_lat}"  # Start point: user's last check-in

    for lat, lon in destinations:
        coordinates += f";{lon},{lat}"

    # Define the Mapbox Matrix API endpoint
    url = f"https://api.mapbox.com/directions-matrix/v1/mapbox/driving/{coordinates}"
    params = {
        "access_token": mapbox_token,
        "annotations": "duration"  # Request travel times
    }

    # Send request to Mapbox API
    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise Exception(f"Mapbox API Error: {response.json()}")

    # Parse the response
    durations = response.json()["durations"][0][1:]  # Travel times (exclude self)
    recommendations["travel_time"] = durations

    # Sort by travel time and get the 10 fastest locations
    fastest_locations = recommendations.sort_values(by="travel_time").head(10)
    return fastest_locations

# Example usage
# Mapbox access token
mapbox_token = "pk.eyJ1IjoiZHVjbmd1eWVubiIsImEiOiJjbTRtejluMWYwMmlpMnRxczdiZ3Npd3F0In0.rTaVkWA3yQWumdPC2b2nKQ"
 

In [55]:
def create_folium_map(user_id, last_checkin, recommendations, file_name="map.html"):
    """
    Create a Folium map showing the user's last check-in location and recommended locations.
    Embed the map in JupyterLab and provide a link to open it in a new tab.

    Parameters:
        user_id (int): The ID of the user.
        last_checkin (DataFrame): Last check-in table with user details.
        recommendations (DataFrame): DataFrame of recommended locations with lat/lon and distances.
        file_name (str): Name of the HTML file to save the map.

    Returns:
        None
    """
    # Get user's last check-in location
    user_checkin = last_checkin.loc[last_checkin["user_id"] == user_id]
    if user_checkin.empty:
        raise ValueError(f"No last check-in data found for user_id: {user_id}")

    user_lat = user_checkin["lat_org"].values[0]
    user_lon = user_checkin["lon_org"].values[0]

    # Initialize Folium map centered at the user's last check-in location
    folium_map = folium.Map(location=[user_lat, user_lon], zoom_start=6)

    # Add user's last check-in location to the map
    folium.Marker(
        location=[user_lat, user_lon],
        popup=f"Last Check-in Location (User {user_id})",
        icon=folium.Icon(color="blue", icon="user")
    ).add_to(folium_map)

    # Add recommended locations to the map
    for _, row in recommendations.iterrows():
        folium.Marker(
            location=[row["lat_org"], row["lon_org"]],
            popup=f"Location ID: {row['location_id']}<br>Distance: {row['distance']:.2f} km",
            icon=folium.Icon(color="green", icon="map-marker")
        ).add_to(folium_map)

    # Save the map to an HTML file
    file_path = os.path.join(os.getcwd(), file_name)
    folium_map.save(file_path)

    # Embed the map in the notebook
    display(IFrame(file_path, width=800, height=600))

    # Provide a clickable link to open the map in a new tab
    display(HTML(f'<a href="{file_path}" target="_blank">Click here to view the map in a new tab</a>'))


In [9]:
 # Group the data by 'state' and calculate the mean of lat_org and lon_org for each cluster
centroids = distinct_location_ids.groupby('state').agg(
    centroid_lat=('lat_org', 'mean'),
    centroid_lon=('lon_org', 'mean')
).reset_index()

# Display the centroids
print(centroids)
mapbox_api_key = "pk.eyJ1IjoiZHVjbmd1eWVubiIsImEiOiJjbTRtejluMWYwMmlpMnRxczdiZ3Npd3F0In0.rTaVkWA3yQWumdPC2b2nKQ"

                 state  centroid_lat  centroid_lon
0               Aargau     47.431627      8.189858
1            Abu Dhabi     24.429192     54.645533
2                Agder     58.360906      8.340660
3            Aguadilla     18.494654    -67.143309
4       Aguascalientes     21.811726   -102.305771
..                 ...           ...           ...
622  Đồng Nai Province     10.732320    107.020889
623     İzmir Province     38.781521     27.004711
624   Łódź Voivodeship     51.761464     19.441567
625              Ōsaka     34.703992    135.509548
626      Žilina Region     49.223666     19.023717

[627 rows x 3 columns]


In [10]:
def find_clusters_along_route(start_point, end_point, centroids, mapbox_api_key, buffer_distance=100000):
    """
    Find clusters along a route between two points using Mapbox Directions API.
    
    Parameters:
    -----------
    start_point : tuple
        (latitude, longitude) of starting point
    end_point : tuple
        (latitude, longitude) of ending point
    centroids : pandas.DataFrame
        DataFrame containing cluster centroids with columns: 
        ['centroid_lat', 'centroid_lon', 'state']
    mapbox_api_key : str
        Mapbox API access token
    buffer_distance : float, optional
        Buffer distance in meters around the route (default: 100000 = 100km)
        
    Returns:
    --------
    list
        List of cluster names (states) that intersect with the route buffer
    dict
        Additional route information including geometry and response data
        Returns None if API request fails
    """
    # Mapbox Directions API endpoint
    url = 'https://api.mapbox.com/directions/v5/mapbox/driving/{},{};{},{}?access_token={}&overview=full&geometries=geojson'
    
    # Format the URL with the coordinates and API key
    formatted_url = url.format(
        start_point[1], start_point[0],  # Switch lat/lon for Mapbox API
        end_point[1], end_point[0],
        mapbox_api_key
    )
    
    try:
        # Request route from Mapbox API
        response = requests.get(formatted_url)
        response.raise_for_status()  # Raise exception for bad status codes
        
        data = response.json()
        route_geometry = data['routes'][0]['geometry']
        route_line = shape(route_geometry)
        
        # Create GeoDataFrame from centroids
        clusters_gdf = gpd.GeoDataFrame(
            centroids,
            geometry=gpd.points_from_xy(centroids['centroid_lon'], centroids['centroid_lat']),
            crs="EPSG:4326"
        )
        
        # Create route buffer
        route_buffer = (gpd.GeoSeries([route_line], crs="EPSG:4326")
                       .to_crs(epsg=3857)
                       .buffer(buffer_distance)
                       .to_crs(epsg=4326))
        
        # Find intersecting clusters
        clusters_gdf = clusters_gdf.set_geometry('geometry')
        intersecting_clusters = clusters_gdf[
            clusters_gdf.geometry.apply(lambda x: x.within(route_buffer[0]))
        ]
        
        # Extract cluster names
        clusters_along_route = intersecting_clusters['state'].tolist()
        
        return clusters_along_route, {
            'route_geometry': route_geometry,
            'route_line': route_line,
            'route_buffer': route_buffer,
            'response_data': data
        }
        
    except requests.exceptions.RequestException as e:
        print(f"Error making API request: {str(e)}")
        return None, None
    except (KeyError, IndexError) as e:
        print(f"Error processing API response: {str(e)}")
        return None, None
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return None, None

In [33]:
def recommend_locations(distinct_location_ids, clusters_along_route, start_point, num_recommendations=10, mapbox_api_key=None):
    """
    Recommends a set of locations based on clusters the route passes through and the fastest travel time.
    
    Args:
    - distinct_location_ids (pd.DataFrame): DataFrame containing location information with columns 
        ['latitude', 'longitude', 'state', 'location_id', 'city', etc.]
    - clusters_along_route (list): List of cluster IDs that the route passes through.
    - start_point (tuple): (latitude, longitude) for the starting point of the route.
    - num_recommendations (int): The maximum number of locations to recommend. Default is 10.
    - mapbox_api_key (str): Mapbox API key for directions API.
    
    Returns:
    - None: Prints the final recommendations and distribution across clusters.
    """
    
    # Step 1: Filter locations in clusters along the route
    filtered_locations = distinct_location_ids[distinct_location_ids['state'].isin(clusters_along_route)]

    # Step 2: Calculate how many locations to pick from each cluster
    num_clusters = len(clusters_along_route)
    base_picks_per_cluster = num_recommendations // num_clusters  # Integer division
    extra_picks = num_recommendations % num_clusters  # Remainder to distribute

    # Create a dictionary to store how many picks per cluster
    cluster_allocation = {cluster: base_picks_per_cluster for cluster in clusters_along_route}
    
    # Distribute remaining picks
    for i in range(extra_picks):
        cluster_allocation[clusters_along_route[i]] += 1

    # Step 3: Select locations from each cluster
    recommendations = pd.DataFrame()
    for cluster in clusters_along_route:
        cluster_locations = filtered_locations[filtered_locations['state'] == cluster]
        if not cluster_locations.empty:
            # Sample the allocated number of locations from this cluster
            num_picks = min(cluster_allocation[cluster], len(cluster_locations))
            picks = cluster_locations.sample(n=num_picks, replace=False)
            recommendations = pd.concat([recommendations, picks], ignore_index=True)

    # Step 4: Calculate fastest travel times using Mapbox API
    def calculate_travel_time(lat, lon):
        url = f"https://api.mapbox.com/directions/v5/mapbox/driving/{start_point[1]},{start_point[0]};{lon},{lat}?access_token={mapbox_api_key}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data['routes'][0]['duration'] / 60  # Convert seconds to minutes
        else:
            return float('inf')  # Return a high value if API fails

    # Add travel times to recommendations
    recommendations['travel_time'] = recommendations.apply(lambda row: calculate_travel_time(row['latitude'], row['longitude']), axis=1)

    # Step 5: For each cluster, select the fastest locations up to its allocation
    final_recommendations = pd.DataFrame()
    for cluster in clusters_along_route:
        cluster_recommendations = recommendations[recommendations['state'] == cluster]
        if not cluster_recommendations.empty:
            top_picks = cluster_recommendations.nsmallest(cluster_allocation[cluster], 'travel_time')
            final_recommendations = pd.concat([final_recommendations, top_picks], ignore_index=True)

    # Step 6: Print the final recommendations and distribution across clusters
    print("Final Recommendations:")
    print(final_recommendations[['location_id', 'city', 'state', 'travel_time']])
    
    print("\nDistribution across clusters:")
    print(final_recommendations['state'].value_counts())

    return final_recommendations


In [37]:
def visualize_travel_plan(final_recommendations, start_point, mapbox_api_key, output_file="travel_plan_map.html"):
    """
    Visualizes a travel plan on a map with recommendations and a route using Mapbox API.
    
    Args:
    - final_recommendations (pd.DataFrame): DataFrame with location recommendations containing 
        ['lat_org', 'lon_org', 'city', 'state', 'travel_time'].
    - start_point (tuple): Starting point as a tuple (latitude, longitude).
    - mapbox_api_key (str): Mapbox API key for directions API.
    - output_file (str): Output file name to save the map (default: "travel_plan_map.html").
    
    Returns:
    - folium.Map: The Folium map object with the route and locations.
    """
    
    # Step 1: Initialize Folium map
    m = folium.Map(location=start_point, zoom_start=6)

    # Step 2: Add markers for recommendations
    for _, row in final_recommendations.iterrows():
        folium.Marker(
            location=(row['lat_org'], row['lon_org']),
            popup=f"City: {row['city']}<br>Cluster: {row['state']}<br>Travel Time: {row['travel_time']} mins",
            tooltip=row['city']
        ).add_to(m)

    # Step 3: Draw the travel plan using Mapbox Directions API
    coordinates = ";".join([f"{row['lon_org']},{row['lat_org']}" for _, row in final_recommendations.iterrows()])
    url = f"https://api.mapbox.com/directions/v5/mapbox/driving/{coordinates}?access_token={mapbox_api_key}&overview=full&geometries=geojson"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        route_geometry = data['routes'][0]['geometry']  # GeoJSON format
        line = LineString(route_geometry['coordinates'])  # Convert to Shapely LineString
        
        # Add route line to the map
        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in line.coords],
            color="blue",
            weight=4,
            opacity=0.7,
            tooltip="Travel Plan"
        ).add_to(m)
    else:
        print(f"Error fetching route: {response.status_code} - {response.text}")

    # Step 4: Save and display the map
    m.save(output_file)
    
    # Return the map for further manipulation or display
    return m


In [58]:
def visualize_travel_plan(final_recommendations, start_point, mapbox_api_key, output_file="travel_plan_map.html"):
    """
    Visualizes a travel plan on a map with recommendations and a route using Mapbox API.
    
    Args:
    - final_recommendations (pd.DataFrame): DataFrame with location recommendations containing 
        ['lat_org', 'lon_org', 'city', 'state', 'travel_time'].
    - start_point (tuple): Starting point as a tuple (latitude, longitude).
    - mapbox_api_key (str): Mapbox API key for directions API.
    - output_file (str): Output file name to save the map (default: "travel_plan_map.html").
    
    Returns:
    - None: Displays the map in the notebook and provides a link to open it in a new tab.
    """
    
    # Step 1: Initialize Folium map
    m = folium.Map(location=start_point, zoom_start=6)

    # Step 2: Add markers for recommendations
    for _, row in final_recommendations.iterrows():
        folium.Marker(
            location=(row['lat_org'], row['lon_org']),
            popup=f"City: {row['city']}<br>Cluster: {row['state']}<br>Travel Time: {row['travel_time']} mins",
            tooltip=row['city']
        ).add_to(m)

    # Step 3: Draw the travel plan using Mapbox Directions API
    coordinates = ";".join([f"{row['lon_org']},{row['lat_org']}" for _, row in final_recommendations.iterrows()])
    url = f"https://api.mapbox.com/directions/v5/mapbox/driving/{coordinates}?access_token={mapbox_api_key}&overview=full&geometries=geojson"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        route_geometry = data['routes'][0]['geometry']  # GeoJSON format
        line = LineString(route_geometry['coordinates'])  # Convert to Shapely LineString
        
        # Add route line to the map
        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in line.coords],
            color="blue",
            weight=4,
            opacity=0.7,
            tooltip="Travel Plan"
        ).add_to(m)
    else:
        print(f"Error fetching route: {response.status_code} - {response.text}")

    # Step 4: Save the map to an HTML file
    m.save(output_file)

    # Step 5: Display the map in the notebook and provide a link to open in a new tab
    display(IFrame(output_file, width=800, height=600))  # Embed map in the notebook
    display(HTML(f'<a href="{output_file}" target="_blank">Click here to view the map in a new tab</a>'))


### Problem 1 test

In [10]:
user_id = 2
recommendations = recommend_unvisited_locations(user_id, last_checkin, distinct_user_location, distinct_location_ids)
recommendations

,location_id,lat_org,lon_org,distance
245200,857571,34.043509,-118.266380,0.089695
49636,1239291,34.043663,-118.266390,0.100292
179070,1326103,34.043968,-118.266604,0.116562
74800,1278649,34.044000,-118.267691,0.118991
60910,255963,34.043105,-118.265716,0.133339
245199,61904,34.042322,-118.265960,0.135100
257639,1285612,34.042509,-118.265765,0.140596
75006,41905,34.044214,-118.266352,0.151596
49629,1324441,34.044034,-118.266049,0.151818
49626,1275017,34.043868,-118.265816,0.155313


In [11]:
# Get the 10 fastest locations
fastest_locations_1 = get_fastest_locations(user_id=user_id, last_checkin=last_checkin, 
                                          recommendations=recommendations, 
                                          mapbox_token=mapbox_token)
fastest_locations_1

,location_id,lat_org,lon_org,distance,travel_time
74800,1278649,34.044000,-118.267691,0.118991,133.9
179070,1326103,34.043968,-118.266604,0.116562,170.2
49630,1319661,34.044420,-118.266359,0.171566,177.3
49636,1239291,34.043663,-118.266390,0.100292,177.8
75006,41905,34.044214,-118.266352,0.151596,177.9
245200,857571,34.043509,-118.266380,0.089695,178.5
291937,1323531,34.044306,-118.266134,0.170837,184.8
49629,1324441,34.044034,-118.266049,0.151818,188.3
49996,147867,34.044396,-118.265853,0.194203,192.7
3885,14709,34.044324,-118.265848,0.188242,193.2


In [12]:
# Create the Folium map
user_map = create_folium_map(user_id=user_id, last_checkin=last_checkin, recommendations=fastest_locations_1)

# Save the map to an HTML file or display
user_map


### Problem 2 test

In [64]:
# Example usage
user_id = 2
two_hop_recommendations = recommend_unvisited_locations_2hop_friends(user_id, last_checkin, distinct_user_location, distinct_location_ids, two_hop_locations)
two_hop_recommendations

,location_id,lat_org,lon_org,distance
245200,857571,34.043509,-118.266380,0.089695
49636,1239291,34.043663,-118.266390,0.100292
74800,1278649,34.044000,-118.267691,0.118991
60910,255963,34.043105,-118.265716,0.133339
245199,61904,34.042322,-118.265960,0.135100
257639,1285612,34.042509,-118.265765,0.140596
75006,41905,34.044214,-118.266352,0.151596
49626,1275017,34.043868,-118.265816,0.155313
291937,1323531,34.044306,-118.266134,0.170837
3885,14709,34.044324,-118.265848,0.188242


In [14]:
# Get the 10 fastest locations
fastest_locations_2 = get_fastest_locations(user_id=user_id, last_checkin=last_checkin, 
                                          recommendations=two_hop_recommendations, 
                                          mapbox_token=mapbox_token)
fastest_locations_2

,location_id,lat_org,lon_org,distance,travel_time
74800,1278649,34.044000,-118.267691,0.118991,133.9
49636,1239291,34.043663,-118.266390,0.100292,177.8
75006,41905,34.044214,-118.266352,0.151596,177.9
245200,857571,34.043509,-118.266380,0.089695,178.5
291937,1323531,34.044306,-118.266134,0.170837,184.8
49996,147867,34.044396,-118.265853,0.194203,192.7
3885,14709,34.044324,-118.265848,0.188242,193.2
49626,1275017,34.043868,-118.265816,0.155313,196.3
60780,39148,34.044009,-118.265300,0.203357,225.3
60910,255963,34.043105,-118.265716,0.133339,235.0


In [22]:
# Create the Folium map
user_map_2_hop_friend = create_folium_map(user_id=user_id, last_checkin=last_checkin, recommendations=fastest_locations_2)

# Save the map to an HTML file or display
user_map_2_hop_friend

### Problem 3 test

In [34]:
# Example usage:
start_point = (21.0285, 105.8542)  # Hanoi
end_point = (10.8231, 106.6297)    # Ho Chi Minh City
clusters, route_info = find_clusters_along_route(
    start_point=start_point,
    end_point=end_point,
    centroids=centroids,
    mapbox_api_key = mapbox_api_key)

if clusters:
    print("Clusters along the route:", clusters)
else:
    print("Failed to find clusters along route")

Clusters along the route: ['Bà Rịa–Vũng Tàu Province', 'Bình Dương Province', 'Bình Thuận Province', 'Hanoi', 'Ho Chi Minh', 'Lâm Đồng Province', 'Tiền Giang', 'Vĩnh Long Province', 'Đồng Nai Province']


In [35]:
final_recommendations = recommend_locations(distinct_location_ids, clusters_along_route, start_point, num_recommendations=10, mapbox_api_key=mapbox_api_key)

Final Recommendations:
   location_id         city                     state  travel_time
0       967022     Vũng Tàu  Bà Rịa–Vũng Tàu Province  1629.645567
1       959186     Vũng Tàu  Bà Rịa–Vũng Tàu Province  1629.645567
2      1459610  Thủ Dầu Một       Bình Dương Province  1652.845700
3       566925    Thuận Nam       Bình Thuận Province  1497.995967
4      1102267      Đống Đa                     Hanoi    14.881417
5       989281     Quận Một               Ho Chi Minh  1606.160417
6      1124608       Ðà Lạt         Lâm Đồng Province  1461.619917
7       919268     Tân Hiệp                Tiền Giang  1678.216800
8      1233719    Vĩnh Long        Vĩnh Long Province  1723.930333
9      1739585   Long Thành         Đồng Nai Province  1568.741933

Distribution across clusters:
state
Bà Rịa–Vũng Tàu Province    2
Bình Dương Province         1
Bình Thuận Province         1
Hanoi                       1
Ho Chi Minh                 1
Lâm Đồng Province           1
Tiền Giang             

In [38]:
m = visualize_travel_plan(final_recommendations, start_point, mapbox_api_key)

# Optionally, display the map in Jupyter Notebook (if working in a notebook environment)
m

# System execution

In [69]:
def choose_problem():
    """
    Display a command-line interface to choose which problem to solve.
    """
    print("Choose a problem to solve:")
    print("1. Recommend a list of 10 unvisited locations for any userID as input.")
    print("2. Recommend a list of 10 unvisited locations for any userID based on 2-hop friends.")
    print("3. Provide a travel plan containing up to 10 locations with the shortest travel time.")
    
    choice = input("Enter the number of the problem you want to solve (1, 2, or 3): ")
    
    if choice == "1":
        solve_problem_1()
    elif choice == "2":
        solve_problem_2()
    elif choice == "3":
        solve_problem_3()
    else:
        print("Invalid choice! Please enter 1, 2, or 3.")
        sys.exit(1)

def solve_problem_1():
    """
    Solve problem 1: Recommend a list of 10 unvisited locations based on travel time.
    """
    user_id = int(input("Enter user ID: "))

    recommendations = recommend_unvisited_locations(user_id, last_checkin, distinct_user_location, distinct_location_ids)
    fastest_locations_1 = get_fastest_locations(user_id=user_id, last_checkin=last_checkin, 
                                                recommendations=recommendations, mapbox_token=mapbox_token)
    
    # Print the results
    print(fastest_locations_1)
    
    # Ask if the user wants to see the map
    show_map = input("Do you want to see the map? (yes/no): ").strip().lower()
    if show_map == "yes":
        user_map = create_folium_map(user_id=user_id, last_checkin=last_checkin, recommendations=fastest_locations_1)
        user_map
    else:
        print("Map display skipped.")

def solve_problem_2():
    """
    Solve problem 2: Recommend a list of 10 unvisited locations based on 2-hop friends.
    """
    user_id = int(input("Enter user ID: "))

    # You should already have the two_hop_locations data
    two_hop_recommendations = recommend_unvisited_locations_2hop_friends(user_id , last_checkin, distinct_user_location, distinct_location_ids, two_hop_locations)

    # Get the 10 fastest locations
    fastest_locations_2 = get_fastest_locations(user_id=user_id, last_checkin=last_checkin, 
                                              recommendations=two_hop_recommendations, 
                                              mapbox_token=mapbox_token)
    print(fastest_locations_2)
    
    # Ask if the user wants to see the map
    show_map = input("Do you want to see the map? (yes/no): ").strip().lower()
    if show_map == "yes":
        user_map_2_hop_friend = create_folium_map(user_id=user_id, last_checkin=last_checkin, recommendations=fastest_locations_2)
        user_map_2_hop_friend
    else:
        print("Map display skipped.")

def solve_problem_3():
    """
    Solve problem 3: Provide a travel plan for up to 10 locations with shortest travel time.
    """
    start_lat, start_lon = input("Enter the starting point (latitude,longitude): ").split(',')
    start_point = (float(start_lat), float(start_lon))  # Convert to tuple of floats
    
    end_lat, end_lon = input("Enter the end point (latitude,longitude): ").split(',')
    end_point = (float(end_lat), float(end_lon))  # Convert to tuple of floats

    clusters, route_info = find_clusters_along_route(
        start_point=start_point,
        end_point=end_point,
        centroids=centroids,
        mapbox_api_key=mapbox_api_key)
    
    if clusters is None:
        print("Failed to find clusters along route")
        return
        
    final_recommendations = recommend_locations(distinct_location_ids, clusters, start_point, num_recommendations=10, mapbox_api_key=mapbox_api_key)

    # Ask if the user wants to see the map
    show_map = input("Do you want to see the map? (yes/no): ").strip().lower()
    if show_map == "yes":
        travel_map = visualize_travel_plan(final_recommendations, start_point, mapbox_api_key)
        travel_map
    else:
        print("Map display skipped.")

if __name__ == "__main__":
    # Example Mapbox token (replace with your actual Mapbox API key)
    mapbox_token = "pk.eyJ1IjoiZHVjbmd1eWVubiIsImEiOiJjbTRtejluMWYwMmlpMnRxczdiZ3Npd3F0In0.rTaVkWA3yQWumdPC2b2nKQ"
    
    # Start the command-line interface
    choose_problem()

Choose a problem to solve:
1. Recommend a list of 10 unvisited locations for any userID as input.
2. Recommend a list of 10 unvisited locations for any userID based on 2-hop friends.
3. Provide a travel plan containing up to 10 locations with the shortest travel time.


Enter the number of the problem you want to solve (1, 2, or 3):  3
Enter the starting point (latitude,longitude):  21.0285, 105.8542
Enter the end point (latitude,longitude):  10.8231, 106.6297


Final Recommendations:
   location_id         city                     state  travel_time
0       967022     Vũng Tàu  Bà Rịa–Vũng Tàu Province  1629.645567
1      1376657     Vũng Tàu  Bà Rịa–Vũng Tàu Province  1629.645567
2      1459610  Thủ Dầu Một       Bình Dương Province  1652.845700
3      1572250   Phan Thiết       Bình Thuận Province  1484.089067
4      1159656     Cầu Giấy                     Hanoi    24.692333
5      1356773     Quận Một               Ho Chi Minh  1606.160417
6      1116672       Ðà Lạt         Lâm Đồng Province  1461.619917
7       919268     Tân Hiệp                Tiền Giang  1678.216800
8       478140    Vĩnh Long        Vĩnh Long Province  1723.930333
9      1739585   Long Thành         Đồng Nai Province  1568.741933

Distribution across clusters:
state
Bà Rịa–Vũng Tàu Province    2
Bình Dương Province         1
Bình Thuận Province         1
Hanoi                       1
Ho Chi Minh                 1
Lâm Đồng Province           1
Tiền Giang             

Do you want to see the map? (yes/no):  yes
